# Stage 2 at update 200 — forward drift and current velocity

Select an **A100 80 GB** runtime and choose **Runtime → Run all**. Mount the Drive containing `MyDrive/MRIxFields2026`. The two retrospective v7 fits use one initialization and the same endpoint, time and noise draws, optimizer settings and auxiliary losses. Both step-200 checkpoints are saved. The historical 100k promotion gate is disabled for this fixed-update experiment.

The primary result is **mean calibrated whole-volume target nRMSE for current velocity minus forward drift** across the frozen 59 development directions of P:0006. Negative favors current velocity. The original 3T → 7T T1w pilot is reported separately. Both evaluations use native source values outside the latent core, four BF16 Heun steps, the correct target decoder and renderer, and matched calibration. Zero transport, wrong field, photometry only and calibrated Stage 1 are scored against the same acquired target. Each arm’s `val_sb` is reported against its own regression target. P:0009 stays reserved.

Results are written under `MyDrive/MRIxFields2026/UnifiedStage2_1ca2b4a_01/stage2_matched_objectives_step200/fd147dd1d06ccbbda4326053a0488d69ee658680/run-0001/`. **Run all** again on a compatible runtime to reuse verified completed fits and directions. The worker prints both checkpoint paths and the final `evaluation/summary.json`.

In [ ]:
import subprocess
from google.colab import drive

gpu = subprocess.check_output(
    ["nvidia-smi", "--id=0", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True
).strip()
name, memory_mib = gpu.rsplit(",", 1)
if "A100" not in name or int(memory_mib.strip()) < 75 * 1024:
    raise RuntimeError(f"Select an A100 80 GB runtime. Assigned GPU: {gpu} MiB")
drive.mount("/content/drive")

In [ ]:
import json
import os
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

commit = "fd147dd1d06ccbbda4326053a0488d69ee658680"
remote = "https://github.com/GuillermoTafoya/MRIxFields.git"
with TemporaryDirectory(prefix="fieldbridge-matched-code-", dir="/content") as checkout:
    repo = Path(checkout)
    def git(*args):
        return subprocess.check_output(["git", "-C", checkout, *args], text=True).strip()

    git("init", "--quiet")
    git("fetch", "--quiet", "--depth=1", remote, commit)
    git("checkout", "--quiet", "--detach", "FETCH_HEAD")
    if git("rev-parse", "HEAD") != commit:
        raise RuntimeError("Implementation commit pin failed.")
    env = dict(os.environ, FIELDBRIDGE_IMPLEMENTATION_COMMIT=commit, CUBLAS_WORKSPACE_CONFIG=":4096:8",
               PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1",
               PYTHONPATH=os.pathsep.join([str(repo / "src"), str(repo / "notebooks")]))
    cmd = [sys.executable, "-B", str(repo / "notebooks/stage2_matched_objective_operator.py")]
    with subprocess.Popen(cmd, cwd=repo, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, errors="replace", bufsize=1) as worker:
        try:
            for line in worker.stdout:
                if line.startswith('{"contract_version": "stage2-unified-term-history-v7"'):
                    row = json.loads(line)
                    if row["step"] == 1 or row["step"] % 10 == 0:
                        print(f"Fit update {row['step']}/200", flush=True)
                else:
                    print(line, end="", flush=True)
        except BaseException:
            worker.terminate()
            raise
        if worker.wait():
            raise subprocess.CalledProcessError(worker.returncode, cmd)